## 1. Importing Necessary Libraries
This initial cell imports all the required Python libraries for the script. This includes os for file system operations, scipy for loading data from MATLAB's .mat files, matplotlib for plotting, numpy for numerical operations, and several key components from the pynwb library for creating and managing the Neurodata Without Borders (NWB) file.

### Attention:
Bear in mind that once nwb objects (unit data or behavior) are defined they cannot be modified and will cause errors if attempting to re-run the cell. Thus I advice restarting the kernel and running the notebook from the top,

In [1]:
import os
import scipy
import matplotlib.pyplot as plt
import numpy as np
from pynwb import NWBFile, NWBHDF5IO
from datetime import datetime
import pytz
from pynwb import NWBFile, NWBHDF5IO
from datetime import datetime
import pytz
from pynwb.ecephys import ElectrodeGroup

## 2. Defining File Paths and Session Names
Useful if needed to process all sessions at once. 

In [2]:
directory = r"\\research-cifs.nyumc.org\research\buzsakilab\Homes\voerom01\TES\TES_superResponders"

names_w_tracking_ripples = ['TES_sResp_M01_20240312',"TES_sResp_M01_20240313",'TES_sResp_M01_20240314',
                            'TES_sResp_M02_20240312','TES_sResp_M02_20240313','TES_sResp_M02_20240314',
                            "TES_sResp_M03_20240621","TES_sResp_M03_20240622","TES_sResp_M03_20240623",
                            "TES_sResp_M05_20240729","TES_sResp_M05_20240730","TES_sResp_M05_20240731"]

names_w_tracking_ripples_maze_change = ['TES_sResp_M01_20240318','TES_sResp_M02_20240318','TES_sResp_M03_20240624']

name = names_w_tracking_ripples[11]

## 3. Loading, Processing, and Organizing Data
This cell will load all data using cell explorer format and organize the variables we want to export as nwb

In [3]:
print(name)

os.chdir(directory+"\\"+name[0:13]+"\\"+name)

tracking_behavior = scipy.io.loadmat(name+'.Tracking.Behavior.mat',simplify_cells=True)
variable_names = list(tracking_behavior)

#
name_variable = list(tracking_behavior)[-1]
if any("tracking" in var for var in variable_names):
    tracking_behavior = tracking_behavior['tracking']
if any("behavior" in var for var in variable_names):
    tracking_behavior = tracking_behavior['behavior']

linearized_position = tracking_behavior['position']['x']
speed = tracking_behavior['Speed']

# Get the sampling rate from the position data (if uniformly sampled)
sampling_rate = tracking_behavior['samplingRate']
x_position = tracking_behavior['position']['x']
y_position = tracking_behavior['position']['y']
position_data = np.array([x_position,y_position]).T
timestamps_data = tracking_behavior['timestamps']
speed_data = tracking_behavior['Speed']
# get cells and ids

cell_info = scipy.io.loadmat(name+'.cell_metrics.cellinfo.mat',simplify_cells=True)['cell_metrics']

cell_types = cell_info['putativeCellType']

# check bad cells

if "tags" in cell_info:
    tags = cell_info["tags"]
    if 'Bad' in list(tags):
        bad_cells = cell_info["tags"]['Bad']
    else:
        bad_cells = []
        
    cell_id = cell_info["cellID"]
    good_cells = np.setdiff1d(cell_id,bad_cells)-1
else:
    good_cells = cell_info["cellID"]-1

cell_types = cell_types[good_cells]

spike_times_all = cell_info["spikes"]["times"][good_cells]
shank_id_all = cell_info['shankID'][good_cells]

if name[10:13] == 'M01':

    rsc_shanks = [3,6]
    ca1_shanks = [2,5]
    ca3_shanks = [1,4]
    
if name[10:13] == 'M02':

    rsc_shanks = [3,5]
    ca1_shanks = [2,4,6]
    ca3_shanks = [1]

if name[10:13] == 'M03':

    rsc_shanks = [5,6]
    ca1_shanks = [2,3,4]
    ca3_shanks = [1]
    
if name[10:13] == 'M05':

    rsc_shanks = [4,6,8]
    ca1_shanks = [3,5,7]
    ca3_shanks = [1,2]
    

cell_area = []
for x in range(cell_types.shape[0]):
    if np.isin(shank_id_all[x],ca1_shanks):
        cell_area.append("CA1")
    if np.isin(shank_id_all[x],ca3_shanks):
        cell_area.append("CA3")
    if np.isin(shank_id_all[x],rsc_shanks):
        cell_area.append("RSC")
cell_area = np.array(cell_area)

pyramidal_cells_rsc = (cell_types != 'Narrow Interneuron') * (cell_area == "RSC")
pyramidal_cells_ca1 = (cell_types == 'Pyramidal Cell') * (cell_area == "CA1")
pyramidal_cells_ca3 = (cell_types == 'Pyramidal Cell') * (cell_area == "CA3")

interneurons_ca1 = (cell_types != 'Pyramidal Cell') * (cell_area == "CA1")
interneurons_ca3 = (cell_types != 'Pyramidal Cell') * (cell_area == "CA3")
interneurons_rsc = (cell_types == 'Narrow Interneuron') * (cell_area == "RSC")

os.chdir(r'C:\Users\GONZAJ81\Downloads')
y_chan_coordinates = scipy.io.loadmat(name[0:14]+'kilosortChanMap.mat',simplify_cells=True)["ycoords"]
x_chan_coordinates = scipy.io.loadmat(name[0:14]+'kilosortChanMap.mat',simplify_cells=True)["xcoords"]
cell_coordinates = scipy.io.loadmat(name[0:14]+'kilosortChanMap.mat',simplify_cells=True)
max_ch_amp = cell_info["maxWaveformCh"][good_cells]
y_cell_coordinates = y_chan_coordinates[max_ch_amp]
x_cell_coordinates = x_chan_coordinates[max_ch_amp]

firing_rates = cell_info['firingRate'][good_cells]
ab_ratio = cell_info['ab_ratio'][good_cells]
acg = cell_info['acg']['wide'][good_cells]
acg_t = cell_info['acg']['narrow'].T
acg = acg_t[good_cells]
burstIndex_Mizuseki2012 = cell_info['burstIndex_Mizuseki2012'][good_cells]
cv2 = cell_info['cv2'][good_cells]
maxWaveformCh = cell_info['maxWaveformCh'][good_cells]
troughToPeak = cell_info['troughToPeak'][good_cells]
waveforms = cell_info['waveforms']['raw'][good_cells]
acg_tau_decay = cell_info['acg_tau_decay'][good_cells]
acg_tau_rise = cell_info['acg_tau_rise'][good_cells]
thetaModulationIndex = cell_info['thetaModulationIndex'][good_cells]
y_position_probe = y_cell_coordinates
x_position_probe = x_cell_coordinates

metrics_2_add = [spike_times_all,cell_types,cell_area,firing_rates,ab_ratio,acg,burstIndex_Mizuseki2012,cv2,maxWaveformCh,troughToPeak,waveforms,acg_tau_decay,acg_tau_rise,thetaModulationIndex, x_position_probe, y_position_probe]

# load single LFP channel 
os.chdir(directory+"\\"+name[0:13]+"\\"+name)

sleep_states_data = scipy.io.loadmat(name+'.SleepState.states.mat',simplify_cells=True,mat_dtype = True)['SleepState']
nrem_states = sleep_states_data["ints"]["NREMstate"]
rem_states = sleep_states_data["ints"]["REMstate"]
wake_states = sleep_states_data["ints"]["WAKEstate"]

ripple_events = scipy.io.loadmat(name+'.ripples.events.mat',simplify_cells=True,mat_dtype = True)['ripples']
ripple_times = ripple_events['timestamps']

lfp_ripple_channel = ripple_events['detectorinfo']['detectionparms']['lfp']
srate_lfp = 1250
lfp_timestamps = np.arange(0,lfp_ripple_channel.shape[0]/srate_lfp,1/srate_lfp)

TES_sResp_M05_20240731


In [26]:

timestamps_data = np.sort(timestamps_data)


## 4. Creating the NWB File Object
This is the first step in building the NWB file. An NWBFile object is instantiated, serving as the main container for all experimental data and metadata. Essential metadata, such as the session description, start time, experimenter, and lab, are provided here.

In [27]:
# 1. Create NWBFile (Metadata is crucial)

session_start_time = datetime(2024, 7, 31, 10, 0, 0, tzinfo=pytz.utc)
nwbfile = NWBFile(
    session_description='Novel Maze Session with Pre and Post sleep',
    identifier='name',
    session_start_time=session_start_time,
    experimenter='Mihály Vöröslakos',
    lab='Buzsáki Lab',
    institution='NYU',
    # Add other required fields...
)

# weeks m01 6-15, m02 8-15, m03 6-15, M05 8-15
from pynwb.file import Subject
# 2. Add Subject metadata directly
nwbfile.subject = Subject(
    subject_id='M05',
    species='Mus musculus',
    strain='C57BL/6J',
    sex='F',
    age='P8W/P15W',      # Estimated age range for 21g female
    #weight='0.021 kg',
    description='Wild-type mouse'
)

## 5. Creating the Behavior Processing Module
NWB uses "processing modules" to organize derived data. This cell creates a module named behavior to group all processed behavioral data, such as the animal's position and speed, which will be added in the following steps.

In [28]:
# Create the processing module for behavior data
behavior_module = nwbfile.create_processing_module(
    name='behavior', 
    description='Processed behavioral data including position and speed.'
)

from pynwb.behavior import Position, SpatialSeries

# Assuming you have the following variables loaded:
# position_timestamps: 1D NumPy array of timestamps (in seconds)
# position_data: 2D NumPy array, shape (time_points, 2) where the 2 is (X, Y)
# speed_data: 1D NumPy array, shape (time_points,) derived from position

# Create the Position container
position = Position(name='AnimalPosition')

# Create the SpatialSeries for the (X, Y) coordinates
position_series = SpatialSeries(
    name='Position',
    data=position_data,
    timestamps=timestamps_data,
    reference_frame='(0,0) is center of arena',
    unit='centimeters', # Always include a unit
    conversion=0.01 # Conversion factor to meters (if not already in meters)
)

# Add the SpatialSeries to the Position container
position.add_spatial_series(position_series)

# Add the Position container to the Processing module
behavior_module.add(position)

from pynwb.base import TimeSeries


# Create the TimeSeries for speed
speed_series = TimeSeries(
    name='Speed',
    data=speed_data,
    timestamps=timestamps_data,
    unit='cm/s',
    description='Instantaneous speed calculated from 2D position data.'
)

# 3. ADD the TimeSeries object to the ProcessingModule
behavior_module.add(speed_series)

Data type,float64
Shape,"(65115,)"
Array size,508.71 KiB
Data type,float64
Shape,"(65115,)"
Array size,508.71 KiB


## 6. Adding Sleep and Ripple Event Intervals
This section handles time-based event data. First, it loads .mat files containing the start and end times for different sleep states (NREM, REM, WAKE) and for sharp-wave ripple (SWR) events. It then creates a TimeIntervals table named SleepStates and populates it by adding a row for each detected event, including its start time, stop time, and a descriptive label (e.g., 'NREM', 'Ripple')

In [29]:
from pynwb.epoch import TimeIntervals

# 1. Initialize the Table
sleep_states = TimeIntervals(
    name='SleepStates', 
    description='Periods of WAKE, NREM, and REM sleep, and Ripple events.'
)
sleep_states.add_column(name='state', description='The behavioral state or event label.')

# 2. Collect all events into a single list
all_events = []

# Helper function to append data to our list
def collect_events(data_source, label):
    for row in data_source:
        all_events.append({
            'start_time': float(row[0]),
            'stop_time': float(row[1]),
            'state': label
        })

#rem_states = rem_states[np.newaxis,:] #if there is a single interval

# Collect from your different variables
collect_events(nrem_states, 'NREM')
collect_events(rem_states, 'REM')
collect_events(wake_states, 'WAKE')
collect_events(ripple_times, 'Ripple')

# 3. SORT the list by start_time (CRITICAL STEP)
# This ensures DANDI validation passes
all_events.sort(key=lambda x: x['start_time'])

# 4. Add the sorted events to the NWB table
for event in all_events:
    sleep_states.add_row(
        start_time=event['start_time'],
        stop_time=event['stop_time'],
        state=event['state']
    )

# 5. Add to the nwbfile
nwbfile.add_time_intervals(sleep_states)

print(f"Total sorted events added: {len(all_events)}")

Total sorted events added: 3669


## 7. Adding Spike Data and Cell Metrics to the Units Table

This is a critical step where all the spike data and associated cell metrics are added to the NWB file. The process involves three main parts:

Defining Hardware Metadata: An ElectrodeGroup is created to provide context about the recording probe.
Defining the Units Table Structure: Custom columns are added to the NWB file's units table using add_unit_column. Each column is given a name and a description, defining the structure that will hold all the detailed metrics for each neuron.
Populating the Table: The code iterates through each neuron, adding a new row to the units table using add_unit(). Each row is populated with the neuron's spike times and all of its corresponding metrics (cell type, firing rate, waveform shape, etc.).

In [30]:
# 2. Add an Electrodes table (essential for units)
# You would get this info from your cell_metrics or a separate file

#nwbfile.add_electrode_group(
#    name='ElectrodeGroup1',
#    description='45 degree insertion targetting hippocampus and neocortex',
#    location='CA3, CA1, RSC',
#    device=device
#)

device = nwbfile.create_device(name='Neuropixels 2.0')

e_group = ElectrodeGroup(
    name='ElectrodeGroup1',
    description='60 degree insertion targetting hippocampus and neocortex',
    location='CA3, CA1, RSC',
    device=device  # Pass the device object here
)
nwbfile.add_electrode_group(e_group)

# --- ADD THIS SECTION BEFORE THE LOOP ---

# A. Non-Waveform Metrics (simple scalars or 1D arrays)
nwbfile.add_unit_column(name='cell_type', description='Cell classification (e.g., Pyramidal, Narrow Interneuron, Wide Interneuron)')
nwbfile.add_unit_column(name='cell_area', description='The brain region or subfield the cell was assigned to (e.g., CA1, CA3, RSC)')
nwbfile.add_unit_column(name='firing_rate', description='Firing rate in Hz: Spike count normalized by the interval between the first and the last spike..')
nwbfile.add_unit_column(name='ab_ratio', description='Waveform asymmetry; the ratio between the two positive peaks (peakB-peakA)/(peakA+peakB).')
nwbfile.add_unit_column(name='burstIndex_Mizuseki2012', description='Burst index as defined by Mizuseki et al. 2012.')
nwbfile.add_unit_column(name='cv2', description='Coefficient of variation (CV_2, 10.1152/jn.1996.75.5.1806).')
nwbfile.add_unit_column(name='maxWaveformCh', description='Max channel zero-indexed: The channel with the largest amplitude.')
nwbfile.add_unit_column(name='troughToPeak', description='Trough-to-peak latency is defined from the trough to the following peak of the waveform.')
nwbfile.add_unit_column(name='acg_tau_decay', description='Decay constant (tau) of the ACG fit.')
nwbfile.add_unit_column(name='acg_tau_rise', description='Rise constant (tau) of the ACG fit.')
nwbfile.add_unit_column(name='thetaModulationIndex', description='Theta modulation index. Originally defined in Cacucci et al., JNeuro 2004. Computed as the difference between the theta modulation trough (defined as mean of autocorrelogram bins, 50-70 msec) and the theta modulation peak (mean of autocorrelogram bins, 100-140 msec) over their sum, scaled from -1 to 1.')
nwbfile.add_unit_column(name='x_position_probe', description='Position along the x-axis for the max amp channel for each cell')
nwbfile.add_unit_column(name='y_position_probe', description='Position along the y-axis for the max amp channel for each cell')

# B. Complex Data (Waveforms and ACG)
# These are typically 1D arrays per unit, so we set the dtype to 'object' 
# to allow arrays of variable length/content (like NumPy arrays) to be stored in the column.
#nwbfile.add_unit_column(name='waveforms', description='Average raw spike waveform from channel with max amplitude.', dtype='object')
#nwbfile.add_unit_column(name='acg', description='Autocorrelogram (ACG) of spike times (wide [-1000 ms : 1 ms: 1000 ms]).', dtype='object')

# 1. WAVEFORMS
nwbfile.add_unit_column(
    name='waveforms', 
    description='Average raw spike waveform from channel with max amplitude.', 
    index=True,
    data=np.array([], dtype='float32').reshape(0, 1) # Initialize with empty data array
)

# 2. ACG
nwbfile.add_unit_column(
    name='acg', 
    description='Autocorrelogram (ACG) of spike times (wide [-1000 ms : 1 ms: 1000 ms]).', 
    index=True,
    data=np.array([], dtype='float32').reshape(0, 1) # Initialize with empty data array
)

# --- MODIFIED LOOP SECTION ---
# 3. Add Units table data
# Assuming cell_metrics is the original dict (used for firing_rate)
# and your new metrics are lists/arrays indexed by unit_i

for unit_i in range(len(spike_times_all)):
    # Get spike times (must be a 1D NumPy array in seconds)
    spike_times_list = spike_times_all[unit_i]
    
    nwbfile.add_unit(
        # Required arguments
        spike_times=spike_times_list,
        id=unit_i + 1,  # Unit IDs start at 1
        
        # --- ADDING YOUR METRICS ---
        
        # Scalar Metrics (from your list 'metrics_2_add')
        cell_type=cell_types[unit_i],
        cell_area=cell_area[unit_i],
        # Note: 'firing_rate' already existed in the original example, 
        # so we keep that structure if possible:
        firing_rate=firing_rates[unit_i], 
        ab_ratio=ab_ratio[unit_i],
        burstIndex_Mizuseki2012=burstIndex_Mizuseki2012[unit_i],
        cv2=cv2[unit_i],
        maxWaveformCh=maxWaveformCh[unit_i],
        troughToPeak=troughToPeak[unit_i],
        acg_tau_decay=acg_tau_decay[unit_i],
        acg_tau_rise=acg_tau_rise[unit_i],
        thetaModulationIndex=thetaModulationIndex[unit_i],
        x_position_probe=x_position_probe[unit_i],
        y_position_probe=y_position_probe[unit_i],

        # Array Metrics (WAVEFORMS and ACG)
        waveforms=waveforms[unit_i][:,np.newaxis],  # Must be a 1D or 2D NumPy array
        acg=acg[unit_i][:,np.newaxis] # Must be a 1D NumPy array
    )

C:\Users\GONZAJ81\AppData\Local\anaconda3\Lib\site-packages\pynwb\file.py:719: UserWarning: Column 'waveforms' is predefined in Units with index=2 which does not match the entered index argument. The predefined index spec will be ignored. Please ensure the new column complies with the spec. This will raise an error in a future version of HDMF.
  self.units.add_column(**kwargs)


In [31]:
# --- Assume these variables are loaded ---
# lfp_data_best_channel: 1D NumPy array of the best LFP trace (shape: time_points,)
# lfp_timestamps: 1D NumPy array of time points (in seconds)
# best_lfp_channel_id: Integer (e.g., 256) - the zero-indexed channel ID with max ripple power
# ----------------------------------------

from pynwb.ecephys import LFP, ElectricalSeries

# 1. Retrieve the existing electrode table
electrode_table = nwbfile.electrodes
# 2. Select the single electrode used for this LFP trace
lfp_channel_indices = np.array([0]) 

# Assuming 'e_group' is already defined from your code block.
import numpy as np

# Set the total number of channels on your probe (e.g., Neuropixels 2.0 = 384)
MAX_PROBE_CHANNELS = 384 

# Get the electrode table object (Correct way, fixing previous error)
electrode_table = nwbfile.electrodes 

# Check if electrodes have already been added (to prevent duplicate additions)
for channel_id in range(MAX_PROBE_CHANNELS):
    nwbfile.add_electrode(
        id=channel_id,
        x=np.nan,  # Placeholder if X/Y/Z are unknown
        y=np.nan,
        z=np.nan,
        imp=np.nan,
        location='CA1',
        filtering='low pass filter', # Standard filtering for spiking data
        group=e_group 
    )
print("Electrode table successfully populated.")


# Now, running the LFP creation code will work:
# electrode_table = nwbfile.electrodes # Already done above
# lfp_channel_indices = np.array([0]) # Or whatever your best channel ID is
# lfp_electrodes = nwbfile.create_electrode_table_region( ... ) # This will now execute

ecephys_module_name = 'ecephys'

ecephys_module = nwbfile.create_processing_module(
        name=ecephys_module_name, 
        description='Contains LFP data.'
    )

# Create the ElectrodeTableRegion (references the single best channel)
lfp_electrodes = nwbfile.create_electrode_table_region(
    region= [0],  # [best_lfp_channel_id]
    description=f'Single channel LFP selected based on highest ripple power.'
)

# 3. Create the ElectricalSeries
# NOTE: The data must be 2D (time_points, channels). Since you have 1 channel, use np.newaxis
lfp_electrical_series = ElectricalSeries(
    name='Best_Ripple_channel_LFP_CA1',
    data=lfp_ripple_channel[:, np.newaxis], # Convert 1D data to 2D (time_points, 1)
    electrodes=lfp_electrodes,
    starting_time=0.0, # Or the actual start time
    rate=1250.0,       # The sampling rate in Hz
    conversion=1e-6,
)

# 4. Add to the Ecephys Module (requires module creation/retrieval first, as shown previously)
# ... (Retrieve/Create ecephys_module) ...
lfp_container = LFP(electrical_series=lfp_electrical_series)
ecephys_module.add(lfp_container)

print(f"LFP from Ripple Channel added successfully.")

Electrode table successfully populated.
LFP from Ripple Channel added successfully.


C:\Users\GONZAJ81\AppData\Local\anaconda3\Lib\site-packages\hdmf\container.py:542: UserWarning: The linked table for DynamicTableRegion 'electrodes' does not share an ancestor with the DynamicTableRegion.
  child._validate_on_set_parent()


## 8. Saving all data into the .nwb file

In [32]:
os.chdir(r'C:\Users\GONZAJ81\OneDrive - NYU Langone Health\Desktop\Subspace Paper Dataset\001695')
# 4. Write the file
with NWBHDF5IO(name+'.nwb', 'w') as io:
    io.write(nwbfile)

print("NWB file created successfully!")

NWB file created successfully!


## 9. Validate the .nwb file 
### This is mandatory for DANDI Upload 
If needed you should install DANDI packages

In [33]:
filename = name+".nwb"

# Use the $ symbol to inject the variable into the shell command
!dandi validate $filename

[DANDI.NON_DANDI_FILENAME] C:\Users\GONZAJ81\OneDrive - NYU Langone Health\Desktop\Subspace Paper Dataset\001695\TES_sResp_M05_20240731.nwb — Filename does not conform to DANDI standard
[DANDI.NON_DANDI_FOLDERNAME] C:\Users\GONZAJ81\OneDrive - NYU Langone Health\Desktop\Subspace Paper Dataset\001695\TES_sResp_M05_20240731.nwb — File is not in folder at root with subject name


2026-01-13 22:29:30,206 [    INFO] Note: NumExpr detected 20 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
2026-01-13 22:29:30,206 [    INFO] NumExpr defaulting to 8 threads.
2026-01-13 22:29:32,782 [    INFO] Logs saved in C:\Users\GONZAJ81\AppData\Local\dandi\dandi-cli\Logs\2026.01.14-03.29.28Z-25392.log


## 10. Organize folders and names
### This is done automatically once you have all the data saved
Note: you need to have created a DANDI set and linked you current folder with the repository

In [34]:
!dandi organize -d organized_folder/ .

2026-01-13 22:32:56,993 [    INFO] Note: NumExpr detected 20 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
2026-01-13 22:32:56,993 [    INFO] NumExpr defaulting to 8 threads.
2026-01-13 22:32:58,067 [    INFO] Loading metadata from 22 files
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of  22 | elapsed:   12.3s remaining:   55.5s
[Parallel(n_jobs=-1)]: Done   7 out of  22 | elapsed:   12.4s remaining:   26.8s
[Parallel(n_jobs=-1)]: Done  10 out of  22 | elapsed:   12.5s remaining:   15.0s
[Parallel(n_jobs=-1)]: Done  13 out of  22 | elapsed:   12.6s remaining:    8.7s
[Parallel(n_jobs=-1)]: Done  16 out of  22 | elapsed:   12.7s remaining:    4.7s
[Parallel(n_jobs=-1)]: Done  19 out of  22 | elapsed:   12.8s remaining:    1.9s
[Parallel(n_jobs=-1)]: Done  22 out of  22 | elapsed:   13.0s finished
2026-01-13 22:33:11,204 [    INFO] Hard link support autodetected; setting files_mode='hardlink'
202

## 10. Final Folder Validation


In [45]:
os.chdir(r'C:\Users\GONZAJ81\Desktop\DANDI set\001695')
#!dandi download https://dandiarchive.org/dandiset/001695/draft

!dandi validate "C:\Users\GONZAJ81\Desktop\DANDI set\001695"


#!dandi validate ./organized_folder
#!cd "organized_folder" && dandi validate .

No errors found.


2026-01-13 23:00:42,279 [    INFO] Note: NumExpr detected 20 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
2026-01-13 23:00:42,279 [    INFO] NumExpr defaulting to 8 threads.
2026-01-13 23:00:59,793 [    INFO] Logs saved in C:\Users\GONZAJ81\AppData\Local\dandi\dandi-cli\Logs\2026.01.14-04.00.40Z-24724.log


In [46]:
import os
os.environ["DANDI_API_KEY"] = "81c798088e03f93948ff6731d1a00a26d25b9fe6"
os.chdir(r'C:\Users\GONZAJ81\Desktop\DANDI set\001695')
!dandi upload

PATH                                                          SIZE     ERRORS  PROGRESS STATUS                MESSAGE                  
dandiset.yaml                                                 2.4 kB                    skipped               should be edited online  
sub-M01/sub-M01_ses-20240308T100000_ecephys.nwb               70.7 MB    0         100% done                                           
sub-M01/sub-M01_ses-20240312T100000_behavior+ecephys.nwb      187.4 MB   0         100% done                                           
sub-M01/sub-M01_ses-20240313T100000_behavior+ecephys.nwb      125.8 MB   0         100% done                                           
sub-M01/sub-M01_ses-20240314T100000_behavior+ecephys.nwb      134.6 MB   0         100% done                                           
sub-M01/sub-M01_ses-20240318T100000_behavior+ecephys.nwb      190.0 MB   0         100% done                                           
sub-M02/sub-M02_ses-20240226T100000_ecephys.nwb 

2026-01-13 23:01:58,011 [    INFO] Note: NumExpr detected 20 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
2026-01-13 23:01:58,011 [    INFO] NumExpr defaulting to 8 threads.
2026-01-13 23:01:58,984 [    INFO] Found 23 files to consider
2026-01-13 23:02:52,816 [    INFO] Logs saved in C:\Users\GONZAJ81\AppData\Local\dandi\dandi-cli\Logs\2026.01.14-04.01.56Z-9360.log


In [41]:
# Replace 001695 with your specific Dandiset ID

# 1. Download the required 'dandiset.yaml' into your organized folder
#!dandi download https://dandiarchive.org/dandiset/001695/draft --download dandiset.yaml --output-dir ./organized_folder

# 2. Upload that folder (it will now know it belongs to 001695)
!dandi upload ./organized_folder

PATH                 SIZE DONE    DONE% CHECKSUM STATUS MESSAGE   
001695\dandiset.yaml                             done   updated   
Summary:                  0 Bytes                1 done 1 updated 
                          <0.00%                                  


2026-01-13 22:51:34,411 [    INFO] Logs saved in C:\Users\GONZAJ81\AppData\Local\dandi\dandi-cli\Logs\2026.01.14-03.51.33Z-14016.log


PATH                                                                           SIZE         ERRORS     PROGRESS STATUS           MESSAGE                  
dandiset.yaml                                                                  2.4 kB                           skipped          should be edited online  
organized_folder/sub-M01/sub-M01_ses-20240308T100000_ecephys.nwb               70.7 MB        1                 ERROR            failed validation        
organized_folder/sub-M01/sub-M01_ses-20240312T100000_behavior+ecephys.nwb      187.4 MB       1                 ERROR            failed validation        
organized_folder/sub-M01/sub-M01_ses-20240313T100000_behavior+ecephys.nwb      125.8 MB       1                 ERROR            failed validation        
organized_folder/sub-M01/sub-M01_ses-20240314T100000_behavior+ecephys.nwb      134.6 MB       1                 ERROR            failed validation        
organized_folder/sub-M01/sub-M01_ses-20240318T100000_behavior+ecephys.

2026-01-13 22:51:39,234 [    INFO] Note: NumExpr detected 20 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
2026-01-13 22:51:39,234 [    INFO] NumExpr defaulting to 8 threads.
2026-01-13 22:51:40,310 [    INFO] Found 23 files to consider
2026-01-13 22:52:00,028 [ WARNING] One or more assets failed validation.  Consult the logfile for details.
2026-01-13 22:52:00,111 [    INFO] Logs saved in C:\Users\GONZAJ81\AppData\Local\dandi\dandi-cli\Logs\2026.01.14-03.51.37Z-13388.log
Error: failed validation
